# 5 · Hallucination filter: stand-alone evaluation

This notebook produces the results of **Section "Hallucination filter"** of the paper:

| Output in this notebook | Paper table / claim |
|---|---|
| Effect of the filter (before/after, hallucination rate, valid lost) | Table `tab:halluc_rates` |
| Confusion matrix of the detector | Table `tab:halluc_confusion` |
| Hallucination rate on principles (20/29) and its breakdown (12 + 8) | In-text + footnote |
| Hallucination rate on non-principle references | In-text |
| The two residual hallucinations (both case law) | In-text |

It also **applies the filter to the raw run-1 XMLs**, producing the post-filter
extractions in `data/xml/extraction_run1_post_filter/`.

## How the filter works (paper, Section "Verification procedure")

For each `<item>` in `<lista_riferimenti_diritto>` the pipeline tries to locate the
reference in the judgment text by, in succession: exact textual match (lowercased,
non-alphanumeric characters removed), URI-based match (URN-NIR / ECLI / CELEX assigned by
Linkoln to both the reference and the judgment text), and number–year window match. For
references of type `princ` a different rule applies: the reference is kept only if it has
a textual match **and** it matches one of the 182 canonical legal principles compiled by
the experts (whitelist matching with normalization and curated aliases, implemented in
`src/principi_classifier.py` + `src/principi_canonici.json`).

## Data

- `data/hallucination/references_test_set.csv`: row-level output of the reference
  matcher for the 50 test judgments (one row per reference parsed by Linkoln inside an
  item; pipe-separated). The `hallucinated` column records **how** the reference was
  located in the judgment text (`urn`, `text match: <pos>`, `window match`,
  `linkoln year number`, ...); an empty value (or the literal `True`) means the reference
  could not be located. Note: in this file references are numbered consecutively across
  the issues of a judgment, while the XMLs in `data/xml/extraction_run1` restart from D1
  in each issue — matching is therefore done on the reference text.
- `data/hallucination/removed_references_ground_truth.csv`: the expert judgment
  (hallucinated or not, with a category and a note) for every reference removed by the
  filter. One row per removed item; `n_documents` counts the individual cited documents
  in the item (one item bundles two fabricated decisions). This file digitizes the expert
  review originally recorded in the working notebook `analyze_hallu_validazione.ipynb`.
- `data/validation_annotator_A1.csv` / `A2.csv`: the post-filter expert annotation
  (notebook 02), whose `Num citazioni estratte presenti nella sentenza` column provides
  the ground truth on the **kept** side.

**Granularity.** The experts count *individual cited documents* (the unit of
Table `tab:cit_counts`: 228 kept citations), while the filter operates on `<item>`
elements. The two are reconciled through `n_documents` in the ground-truth file.

**Scope.** As everywhere in the validation, the 4 LLM-extracted issues marked as absent
by the annotators are excluded; the filter decisions for the items of those issues are
still computed (and applied to the XMLs), but they do not enter the evaluation.

In [1]:
import sys
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

DATA = '../data'
sys.path.append('../src')
from principi_classifier import classify   # canonical-principle whitelist matcher

df = pd.read_csv(f'{DATA}/hallucination/references_test_set.csv', sep='|')
df = df[df['questione_element'] == 'lista_riferimenti_diritto'].copy()
print(f"reference rows: {len(df)} over {df['filename'].nunique()} judgments")

gt = pd.read_csv(f'{DATA}/hallucination/removed_references_ground_truth.csv')
print(f"ground-truth rows (removed references): {len(gt)}")

# The 4 LLM-extracted issues marked as absent by the annotators (notebook 01):
# excluded from the evaluation, as in the rest of the validation.
HALLUCINATED_ISSUES = [
    ('Sentenza_V43_3256_2021.xml', 'Q1'),
    ('Sentenza_Z29_1812_2023.xml', 'Q2'),
    ('Sentenza_V16_372_2022.xml', 'Q1'),
    ('Sentenza_V22_5547_2023.xml', 'Q2'),
]

reference rows: 302 over 50 judgments
ground-truth rows (removed references): 35


## Filter decisions

A row of the matcher output is *verified* when the `hallucinated` column records a match.
An `<item>` is kept when **any** of the references parsed inside it is verified (the
paper's footnote: an item is removed only if all references identified within it are
recognized as hallucinated). For `princ` items the canonical-list condition is added.

In [2]:
df['row_verified'] = ~df['hallucinated'].isin([np.nan, '', 'True'])

items_all = (df.groupby(['filename', 'id_questione', 'reference_id'])
               .agg(reference_type=('reference_type', 'first'),
                    original_text=('original_text', 'first'),
                    text_match=('row_verified', 'any'))
               .reset_index())

is_princ = items_all['reference_type'] == 'princ'
items_all['kept'] = items_all['text_match']
items_all.loc[is_princ, 'kept'] = items_all.loc[is_princ].apply(
    lambda r: bool(r['text_match'] and classify(r['original_text']).exists), axis=1)

# Evaluation set: drop the 4 absent issues
mask_dropped_issue = items_all.apply(
    lambda r: (r['filename'], r['id_questione']) in HALLUCINATED_ISSUES, axis=1)
items = items_all[~mask_dropped_issue].copy()
is_princ_eval = items['reference_type'] == 'princ'

print(f"items (all 78 issues): {len(items_all)}")
print(f"items (74 analysed issues): {len(items)}")
print(f"  principles:     kept {int(items[is_princ_eval]['kept'].sum()):3d}, "
      f"removed {int((~items[is_princ_eval]['kept']).sum()):3d}")
print(f"  non-principles: kept {int(items[~is_princ_eval]['kept'].sum()):3d}, "
      f"removed {int((~items[~is_princ_eval]['kept']).sum()):3d}")

items (all 78 issues): 249
items (74 analysed issues): 229
  principles:     kept   6, removed  23
  non-principles: kept 188, removed  12


In [3]:
# Cross-check: the recomputed removed set must coincide with the ground-truth file
removed = items[~items['kept']][['filename', 'id_questione', 'reference_id']]
removed_set = set(map(tuple, removed.values))
gt_set = set(map(tuple, gt[['filename', 'id_questione', 'reference_id']].values))
assert removed_set == gt_set, (
    f"mismatch: recomputed-only {removed_set - gt_set}, ground-truth-only {gt_set - removed_set}")
print(f"OK: the {len(gt_set)} removed items recomputed from the data coincide "
      f"with data/hallucination/removed_references_ground_truth.csv")

OK: the 35 removed items recomputed from the data coincide with data/hallucination/removed_references_ground_truth.csv


## The kept side: residual hallucinations

For the citations that pass the filter, the ground truth is the post-filter expert
annotation: the experts recorded, per issue, how many extracted citations actually appear
in the judgment text (`Num citazioni estratte presenti nella sentenza`). Counts are at
the document level and use the same combined N=50 frame as notebook 02 (shared judgments
averaged across the two annotators; only issues marked present by both annotators).

In [4]:
COL_PRESENT = 'questione presente nella sentenza (TRUE/FALSE)'
COL_CIT_ESTRATTE = '# citazioni estratte (contare i riferimenti di diritto in J)'
COL_CIT_IN_SENT  = 'Num citazioni estratte presenti nella sentenza'
COL_CIT_PRINCIPI = '# citazioni principi estratte'

df_a = pd.read_csv(f'{DATA}/validation_annotator_A1.csv')   # A1
df_p = pd.read_csv(f'{DATA}/validation_annotator_A2.csv')   # A2
common_sentenze = set(df_a['Sentenza']) & set(df_p['Sentenza'])

# joint-present flag (as in notebooks 02-03)
shared_concat = pd.concat([
    df_a[df_a['Sentenza'].isin(common_sentenze)],
    df_p[df_p['Sentenza'].isin(common_sentenze)],
], ignore_index=True)
shared_concat[COL_PRESENT] = shared_concat[COL_PRESENT].astype(bool)
joint_lookup = shared_concat.groupby(['Sentenza', 'Questioni estratte'])[COL_PRESENT].all()

def with_joint_present(d):
    out = d.copy()
    out['is_present_joint'] = out[COL_PRESENT].astype(bool)
    m = out['Sentenza'].isin(common_sentenze)
    keys = pd.MultiIndex.from_arrays([out.loc[m, 'Sentenza'].values,
                                      out.loc[m, 'Questioni estratte'].values])
    out.loc[m, 'is_present_joint'] = joint_lookup.reindex(keys).values
    return out

df_a, df_p = with_joint_present(df_a), with_joint_present(df_p)

# combined N=50 frame: unique judgments as-is, shared judgments averaged
num_cols = [COL_CIT_ESTRATTE, COL_CIT_IN_SENT, COL_CIT_PRINCIPI]
df_uniq = pd.concat([df_a[~df_a['Sentenza'].isin(common_sentenze)],
                     df_p[~df_p['Sentenza'].isin(common_sentenze)]], ignore_index=True)
df_sh = pd.concat([df_a[df_a['Sentenza'].isin(common_sentenze)],
                   df_p[df_p['Sentenza'].isin(common_sentenze)]], ignore_index=True)
df_sh_avg = (df_sh.assign(is_present_joint=df_sh['is_present_joint'].astype(int))
                  .groupby(['Sentenza', 'Questioni estratte'], as_index=False)
                  [num_cols + ['is_present_joint']].mean())
df_comb = pd.concat([df_uniq[df_uniq['is_present_joint'] == True][['Sentenza'] + num_cols],
                     df_sh_avg[df_sh_avg['is_present_joint'] == 1][['Sentenza'] + num_cols]],
                    ignore_index=True)

n_kept   = df_comb[COL_CIT_ESTRATTE].sum()
n_kept_valid = df_comb[COL_CIT_IN_SENT].sum()
n_kept_halluc = n_kept - n_kept_valid
n_kept_princ = df_comb[COL_CIT_PRINCIPI].sum()
print(f"citations kept by the filter (expert document count): {n_kept:.0f}")
print(f"  of which present in the judgment text: {n_kept_valid:.0f}")
print(f"  residual hallucinations: {n_kept_halluc:.0f}")
print(f"  principles kept: {n_kept_princ:.0f}")

citations kept by the filter (expert document count): 228
  of which present in the judgment text: 226
  residual hallucinations: 2
  principles kept: 6


In [5]:
# The residual hallucinations: issues where the experts found fewer citations in the
# judgment than the filter kept
for d, tag in [(df_a, 'A1'), (df_p, 'A2')]:
    res = d[d[COL_CIT_ESTRATTE] > d[COL_CIT_IN_SENT]]
    for _, r in res.iterrows():
        print(f"{tag}  {r['Sentenza']}  (extracted {r[COL_CIT_ESTRATTE]:.0f}, "
              f"in text {r[COL_CIT_IN_SENT]:.0f})")
        print(f"    kept citations: {str(r['Lista citazioni estratte'])[:200]}".replace(chr(10), '; '))

A1  Sentenza_V12_962_2022.json  (extracted 3, in text 2)
    kept citations: D1 - art. 2479, comma 1, c.c.; D2 - art. 28, comma 4, dlgs n. 175/2014; D3 - Cassazione, sez. trib., n. 3331/2020
A1  Sentenza_V16_372_2022.json  (extracted 2, in text 1)
    kept citations: D1 - Cassazione n. 372/2022; D2 - D.L. n. 83/2012 (obblighi documentali)
A2  Sentenza_V12_962_2022.json  (extracted 3, in text 2)
    kept citations: D1 - art. 2479, comma 1, c.c.; D2 - art. 28, comma 4, dlgs n. 175/2014; D3 - Cassazione, sez. trib., n. 3331/2020


Both residual hallucinations are case-law references (`Cassazione, sez. trib., n. 3331/2020`
and `Cassazione n. 372/2022`) whose year and number match some combination of year and
number found in the judgment, with an incorrect issuing authority — as stated in the paper.

## Effect of the filter — Tables `tab:halluc_rates` and `tab:halluc_confusion`

In [6]:
n_removed        = gt['n_documents'].sum()
n_removed_halluc = gt.loc[gt['hallucinated'], 'n_documents'].sum()
n_removed_valid  = gt.loc[~gt['hallucinated'], 'n_documents'].sum()

n_before        = n_kept + n_removed
n_before_halluc = n_kept_halluc + n_removed_halluc
n_valid         = n_before - n_before_halluc

print('Effect of the hallucination filter (N=50) — Table tab:halluc_rates:')
print(pd.DataFrame({
    'Citations':    [f'{n_before:.0f}', f'{n_kept:.0f}'],
    'Hallucinated': [f'{n_before_halluc:.0f}', f'{n_kept_halluc:.0f}'],
    'Halluc. rate': [f'{n_before_halluc / n_before:.1%}', f'{n_kept_halluc / n_kept:.1%}'],
    'Valid lost':   ['---', f'{n_removed_valid:.0f} ({n_removed_valid / n_valid:.1%} of valid)'],
}, index=['Before filter', 'After filter']).to_string())

print()
print('Confusion matrix of the hallucination detector — Table tab:halluc_confusion:')
print(pd.DataFrame({
    'Flagged':     [f'{n_removed_halluc:.0f}', f'{n_removed_valid:.0f}', f'{n_removed:.0f}'],
    'Not flagged': [f'{n_kept_halluc:.0f}', f'{n_kept_valid:.0f}', f'{n_kept:.0f}'],
    'Total':       [f'{n_before_halluc:.0f}', f'{n_valid:.0f}', f'{n_before:.0f}'],
}, index=['Actual hallucination', 'Not hallucination', 'Total']).to_string())

print()
print('Detector metrics:')
print(f"  precision  (flagged & hallucinated / flagged):    {n_removed_halluc / n_removed:.1%}")
print(f"  recall     (flagged & hallucinated / hallucinated): {n_removed_halluc / n_before_halluc:.1%}")
print(f"  specificity (valid retained / valid):             {n_kept_valid / n_valid:.1%}")

Effect of the hallucination filter (N=50) — Table tab:halluc_rates:
              Citations Hallucinated Halluc. rate         Valid lost
Before filter       264           31        11.7%                ---
After filter        228            2         0.9%  7 (3.0% of valid)

Confusion matrix of the hallucination detector — Table tab:halluc_confusion:
                     Flagged Not flagged Total
Actual hallucination      29           2    31
Not hallucination          7         226   233
Total                     36         228   264

Detector metrics:
  precision  (flagged & hallucinated / flagged):    80.6%
  recall     (flagged & hallucinated / hallucinated): 93.5%
  specificity (valid retained / valid):             97.0%


### Bootstrap confidence intervals on the filter's effect

95% percentile intervals from a cluster bootstrap at the judgment level: the per-judgment
citation counts (kept / kept-valid from the post-filter annotation, removed-hallucinated /
removed-valid from the ground-truth file) are resampled with replacement over the 50
judgments (B=10,000), and the rates of Table `tab:halluc_rates` are recomputed on each
replicate. These are the bracketed intervals reported in the paper.

In [7]:
# per-judgment counts (judgment key: filename without extension)
kept_j = (df_comb.assign(stem=df_comb['Sentenza'].str.replace('.json', '', regex=False))
                 .groupby('stem')[[COL_CIT_ESTRATTE, COL_CIT_IN_SENT]].sum())
rem_j = (gt.assign(stem=gt['filename'].str.replace('.xml', '', regex=False),
                   halluc_docs=gt['n_documents'].where(gt['hallucinated'], 0),
                   valid_docs=gt['n_documents'].where(~gt['hallucinated'], 0))
           .groupby('stem')[['halluc_docs', 'valid_docs']].sum())
J = kept_j.join(rem_j, how='outer').fillna(0)
assert len(J) == 50

kept     = J[COL_CIT_ESTRATTE].to_numpy(float)
kept_ok  = J[COL_CIT_IN_SENT].to_numpy(float)
rem_hal  = J['halluc_docs'].to_numpy(float)
rem_ok   = J['valid_docs'].to_numpy(float)

rng = np.random.default_rng(0)
idx = rng.integers(0, 50, size=(10_000, 50))
K, KO, RH, RO = (a[idx].sum(axis=1) for a in (kept, kept_ok, rem_hal, rem_ok))
rate_before = (K - KO + RH) / (K + RH + RO)     # hallucinated / all extracted
rate_after  = (K - KO) / K                      # residual hallucinated / kept
valid_lost  = RO / (KO + RO)                    # valid removed / all valid

for name, point, boot in [
    ('hallucination rate before filter', n_before_halluc / n_before, rate_before),
    ('hallucination rate after filter',  n_kept_halluc / n_kept,     rate_after),
    ('valid citations lost',             n_removed_valid / n_valid,  valid_lost),
]:
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"{name:38s} {point:6.1%}  [{lo:.1%}, {hi:.1%}]")

hallucination rate before filter        11.7%  [6.6%, 17.8%]
hallucination rate after filter          0.9%  [0.0%, 2.3%]
valid citations lost                     3.0%  [1.1%, 5.3%]


## Hallucinations by citation type (in-text numbers)

For principles, the paper counts as hallucinated both the principles that are not cited in
the judgment and the concepts that do appear in the text but are not recognized legal
principles. The 6 principles kept by the filter are all valid (the two residual
hallucinations are case law), so the principle totals are: 6 kept + 23 removed = 29
references.

In [8]:
princ_gt = gt[gt['reference_type'] == 'princ']
n_princ_removed = princ_gt['n_documents'].sum()
n_princ_halluc  = princ_gt.loc[princ_gt['hallucinated'], 'n_documents'].sum()
n_princ_total   = int(items[is_princ_eval]['kept'].sum()) + n_princ_removed
n_not_a_principle = princ_gt.loc[princ_gt['category'] == 'not_a_principle', 'n_documents'].sum()
n_not_in_text     = princ_gt.loc[princ_gt['category'] == 'principle_not_in_text', 'n_documents'].sum()

print(f"principle references in the test set: {n_princ_total}")
print(f"hallucinated principles: {n_princ_halluc} "
      f"(hallucination rate {n_princ_halluc / n_princ_total:.0%})")
print(f"  concepts in the text that are not principles: {n_not_a_principle}")
print(f"  proper principles not cited in the judgment:  {n_not_in_text}")

n_nonprinc        = n_before - n_princ_total
n_nonprinc_halluc = n_before_halluc - n_princ_halluc
print(f"\nnon-principle references: {n_nonprinc:.0f}")
print(f"hallucinated non-principle references: {n_nonprinc_halluc:.0f} "
      f"(hallucination rate {n_nonprinc_halluc / n_nonprinc:.1%})")

principle references in the test set: 29
hallucinated principles: 20 (hallucination rate 69%)
  concepts in the text that are not principles: 12
  proper principles not cited in the judgment:  8

non-principle references: 235
hallucinated non-principle references: 11 (hallucination rate 4.7%)


## Applying the filter to the XMLs

We apply the item-level decisions (computed on **all** 78 issues) to the raw run-1
extractions, producing `data/xml/extraction_run1_post_filter/`. Removed `<item>`s are
deleted from `<lista_riferimenti_diritto>` together with the corresponding
`<motivo_citazione>` entries; reference ids are left untouched. Since the XMLs number
references per issue while the matcher output numbers them per judgment, items are
matched on (issue id, reference text).

In [9]:
import xml.etree.ElementTree as ET
from pathlib import Path

SRC_XML = Path(f'{DATA}/xml/extraction_run1')
DST_XML = Path(f'{DATA}/xml/extraction_run1_post_filter')
DST_XML.mkdir(exist_ok=True)

decision = {}
for _, r in items_all.iterrows():
    decision[(r['filename'], r['id_questione'], r['original_text'].strip())] = bool(r['kept'])

n_files = n_removed_items = n_unmatched = 0
for f in sorted(SRC_XML.glob('*.xml')):
    tree = ET.parse(f)
    for q in tree.getroot().iter('questione'):
        qid = q.get('id')
        lista = q.find('lista_riferimenti_diritto')
        if lista is None:
            continue
        drop_ids = []
        for it in list(lista.findall('item')):
            key = (f.name, qid, (it.get('ref') or '').strip())
            if key not in decision:
                n_unmatched += 1
                print(f"WARNING: no filter decision for {key}")
                continue
            if not decision[key]:
                lista.remove(it)
                drop_ids.append(it.get('id'))
                n_removed_items += 1
        mot = q.find('motivo_citazione')
        if mot is not None and drop_ids:
            for it in list(mot.findall('item')):
                if it.get('id_ref') in drop_ids:
                    mot.remove(it)
    tree.write(DST_XML / f.name, encoding='utf-8', xml_declaration=True)
    n_files += 1

n_removed_all = int((~items_all['kept']).sum())
print(f"{n_files} XML files written to {DST_XML}")
print(f"{n_removed_items} reference items removed "
      f"({len(gt)} in the 74 analysed issues + "
      f"{n_removed_all - len(gt)} in the 4 excluded issues)")
print(f"unmatched items: {n_unmatched}")

50 XML files written to ../data/xml/extraction_run1_post_filter
38 reference items removed (35 in the 74 analysed issues + 3 in the 4 excluded issues)
unmatched items: 0
